# Regression in the Gifi Framework: Morals

Python equivalent of R Gifi vignette: *Regression in Gifi*

**Morals** (Multiple Optimal Regression with Alternating Least Squares) performs
regression with **optimally scaled** predictors and/or response. Transformations
can be linear, monotone (ordinal), or smooth splines.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pygifi import Morals, get_dataset
from pygifi.utils.splines import knots_gifi

## 1 · Neumann Dataset

We use the Neumann dataset (`n=65`) throughout this vignette. The response variable is `y` (column 0); the predictors are the remaining columns.

In [ ]:
neumann = get_dataset('neumann')
print(neumann.shape)
print(neumann.head())

# Split into response and predictors
y = neumann.iloc[:, 0]
X = neumann.iloc[:, 1:]
print(f'\nResponse: {y.name} ({len(y)} obs)')
print(f'Predictors: {list(X.columns)}')

## 2 · Linear Regression (Unconstrained)

The default Morals call applies numerical (unconstrained) optimal scaling to both predictors and response.

In [ ]:
m_linear = Morals(eps=1e-8, itmax=1000).fit(X, y)
print(m_linear)

In [ ]:
# R² (SMC = Squared Multiple Correlation)
print(f'R² (SMC): {m_linear.result_["smc"]:.4f}')

# Beta coefficients
beta = m_linear.result_['beta'].flatten()
print('\nBeta coefficients:')
for name, b in zip(X.columns, beta):
    print(f'  {name}: {b:.4f}')

In [ ]:
# Fitted vs. actual
yhat = m_linear.result_['yhat']
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y, yhat, alpha=0.6, edgecolors='k', s=40)
lims = [min(y.min(), yhat.min()), max(y.max(), yhat.max())]
ax.plot(lims, lims, 'r--', lw=1)
ax.set_xlabel('Observed y')
ax.set_ylabel('Fitted ŷ')
ax.set_title(f'Morals Linear — R² = {m_linear.result_["smc"]:.3f}')
plt.tight_layout()
plt.show()

---

## 3 · Monotone Regression (Ordinal Response)

Setting `yordinal=True` constrains the response transformation to be **monotone non-decreasing** (isotone), which is appropriate when y is an ordinal measure.

In [ ]:
m_mono = Morals(ydegrees=1, yordinal=True, eps=1e-8, itmax=1000).fit(X, y)
print(m_mono)
print(f'\nR² (SMC): {m_mono.result_["smc"]:.4f}')

In [ ]:
# Check monotonicity of yhat
yhat_mono = m_mono.result_['yhat']
y_sorted_idx = np.argsort(y.values)
yhat_at_sorted_y = yhat_mono[y_sorted_idx]
is_monotone = np.all(np.diff(yhat_at_sorted_y) >= -1e-10)
print(f'yhat is monotone non-decreasing w.r.t. y: {is_monotone}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(y.values[y_sorted_idx], yhat_at_sorted_y, 'o-', ms=4, alpha=0.7)
ax.set_xlabel('Sorted observed y')
ax.set_ylabel('Transformed ŷ (monotone)')
ax.set_title('Morals Monotone — Transformation of y')
plt.tight_layout()
plt.show()

---

## 4 · Polynomial Spline Regression

Setting `xknots` and `xdegrees=2` fits **quadratic B-splines** for each predictor transformation.

In [ ]:
# Generate E-type knots (no interior knots — pure polynomial over data range)
xknots_parsed = [knots_gifi(pd.DataFrame(X.iloc[:, i]), type='E')[0]
                 for i in range(X.shape[1])]
print('Knot vectors:', xknots_parsed)

In [ ]:
m_spline = Morals(xknots=xknots_parsed, xdegrees=2, eps=1e-8, itmax=1000).fit(X, y)
print(m_spline)
print(f'\nR² (SMC): {m_spline.result_["smc"]:.4f}')

In [ ]:
# Compare betas across 3 models
models = {'Linear': m_linear, 'Monotone': m_mono, 'Spline': m_spline}
df_betas = pd.DataFrame(
    {name: m.result_['beta'].flatten() for name, m in models.items()},
    index=X.columns
)
print('\nBeta comparison across Morals variants:')
print(df_betas.round(4))

In [ ]:
# Plot xhat transformations for a spline model
xhat = m_spline.result_['xhat']   # (n, p)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i, (ax, col) in enumerate(zip(axes, X.columns)):
    ax.scatter(X.iloc[:, i], xhat[:, i], alpha=0.6, s=25)
    ax.set_xlabel(f'Original {col}')
    ax.set_ylabel('Optimal transform')
    ax.set_title(f'Spline transform: {col}')
plt.tight_layout()
plt.show()

---

## 5 · Model Comparison Summary

In [ ]:
summary = pd.DataFrame({
    'Model': list(models.keys()),
    'SMC (R²)': [m.result_['smc'] for m in models.values()],
    'Loss': [m.result_['f'] for m in models.values()],
    'Iters': [m.result_['ntel'] for m in models.values()],
})
print(summary.round(4).to_string(index=False))

## 6 · Summary Method

In [ ]:
m_mono.summary()

---

## 7 · References

- De Leeuw, J. (1988). Multivariate analysis with linearizable regressions. *Psychometrika*, 53(4), 437–454.
- Breiman, L., & Friedman, J. H. (1985). Estimating optimal transformations for multiple regression and correlation. *JASA*, 80(391), 580–598.
- Mair, P., De Leeuw, J. (2010). Morals: Multiple optimal regression by alternating least squares. *Journal of Statistical Software*, 31(4).